In [ ]:
import os
import hashlib
import tkinter as tk
from tkinter import filedialog, messagebox

def get_file_hash(file_path):
    """파일의 고유 해시값을 계산합니다."""
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        buf = f.read(65536)
        while len(buf) > 0:
            hasher.update(buf)
            buf = f.read(65536)
    return hasher.hexdigest()

def is_suffix_file(filename):
    """
    파일 이름에 접미사가 있는지 판단합니다.
    일반적으로 숫자가 붙거나 특정 기호가 붙은 경우를 체크합니다.
    """
    name_only = os.path.splitext(filename)[0]
    # 접미사 판단 기준: '_'나 ' ' 뒤에 숫자나 문자가 붙는 경우 등
    # 여기서는 간단하게 '_'나 '-'가 포함되어 있으면 접미사가 있는 것으로 간주할 수 있습니다.
    # 혹은 원본 파일명을 알고 있다면 더 정확하겠지만, 여기서는 '더 짧은 이름'을 원본으로 가정하는 로직을 추가합니다.
    return "_" in name_only or "-" in name_only or "(" in name_only

def clean_duplicate_images():
    root = tk.Tk()
    root.withdraw()
    target_dir = filedialog.askdirectory(title="중복 사진을 정리할 폴더를 선택하세요")

    if not target_dir:
        return

    # 해시값을 키로, 파일 경로 리스트를 값으로 저장
    hashes = {}
    valid_extensions = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

    print(f"분석 시작: {target_dir}")

    for root_path, _, files in os.walk(target_dir):
        for file in files:
            if file.lower().endswith(valid_extensions) and not file.startswith('.'):
                file_path = os.path.join(root_path, file)
                file_hash = get_file_hash(file_path)
                
                if file_hash not in hashes:
                    hashes[file_hash] = []
                hashes[file_hash].append(file_path)

    deleted_count = 0

    # 중복된 해시 뭉치들 분석
    for h, file_list in hashes.items():
        if len(file_list) > 1:
            # 1. 파일 이름 길이를 기준으로 정렬 (짧은 이름이 원본일 확률이 높음)
            # 2. 특수 기호(_, -, () 등)가 없는 파일을 우선순위로 정렬
            file_list.sort(key=lambda x: (is_suffix_file(os.path.basename(x)), len(os.path.basename(x))))
            
            # 첫 번째 파일(가장 깨끗한 이름)을 남기고 나머지는 삭제
            keep_file = file_list[0]
            delete_files = file_list[1:]
            
            for d_file in delete_files:
                try:
                    os.remove(d_file)
                    print(f"[삭제됨] {os.path.basename(d_file)} (원본 보존: {os.path.basename(keep_file)})")
                    deleted_count += 1
                except Exception as e:
                    print(f"[삭제 실패] {d_file}: {e}")

    messagebox.showinfo("완료", f"총 {deleted_count}개의 중복 사진을 삭제했습니다.")

if __name__ == "__main__":
    clean_duplicate_images()

분석 시작: D:/DATA/sample/normal
[삭제됨] origin_2_20200731_140619_000750_1.jpg (원본 보존: origin_2_20200731_140619_000750.jpg)
[삭제됨] origin_2_20200731_140619_000750_2.jpg (원본 보존: origin_2_20200731_140619_000750.jpg)
[삭제됨] origin_2_20200731_140619_000750_3.jpg (원본 보존: origin_2_20200731_140619_000750.jpg)
[삭제됨] origin_2_20200731_140619_000780_1.jpg (원본 보존: origin_2_20200731_140619_000780.jpg)
[삭제됨] origin_2_20200731_140619_000780_2.jpg (원본 보존: origin_2_20200731_140619_000780.jpg)
[삭제됨] origin_2_20200731_140619_000780_3.jpg (원본 보존: origin_2_20200731_140619_000780.jpg)
[삭제됨] origin_2_20200731_140619_000810_1.jpg (원본 보존: origin_2_20200731_140619_000810.jpg)
[삭제됨] origin_2_20200731_140619_000810_2.jpg (원본 보존: origin_2_20200731_140619_000810.jpg)
[삭제됨] origin_2_20200731_140619_000810_3.jpg (원본 보존: origin_2_20200731_140619_000810.jpg)
[삭제됨] origin_2_20200731_140619_000840_1.jpg (원본 보존: origin_2_20200731_140619_000840.jpg)
[삭제됨] origin_2_20200731_140619_000840_2.jpg (원본 보존: origin_2_20200731_140619_0008